# Preprocessing for Irrigation Need Models
This notebook creates pipelines and preprocessors for the models used in the existing notebooks.

In [14]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
import os

RSEED = 50


In [15]:
from pathlib import Path
import os

cwd = Path.cwd()
repo_root = cwd
while repo_root != repo_root.parent and not (repo_root / 'data' / 'playground-series-s6e4' / 'train.csv').exists():
    repo_root = repo_root.parent

data_path = repo_root / 'data' / 'playground-series-s6e4' / 'train.csv'
if not data_path.exists():
    raise FileNotFoundError(f'Cannot find data/playground-series-s6e4/train.csv from {cwd} or any parent directory')

print('Using dataset:', data_path)


Using dataset: /home/sajit/Documents/dataScienceBootCampNeueFische/Ginger_Gradient/week8/ML_Project/ds-ml-project/data/playground-series-s6e4/train.csv


In [11]:
data_train = pd.read_csv(data_path)
data_train_X = data_train.drop(columns=['id', 'Irrigation_Need'])
data_train_y = data_train['Irrigation_Need']
data_train.head()

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


In [16]:
data_train_X = data_train.drop(columns=["id" , "Irrigation_Need"])
data_train_y = data_train["Irrigation_Need"]


In [17]:
df = pd.read_csv(data_path)
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(data_train_X, data_train_y, test_size=0.30, random_state=RSEED,stratify=data_train_y)

X_train.shape 


(441000, 19)

In [5]:
numeric_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()
print('Numerical columns:', numeric_cols)
print('Categorical columns:', cat_cols)


Numerical columns: ['Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm']
Categorical columns: ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']


In [6]:
scaling_transformation = ColumnTransformer([('scaling', MinMaxScaler(), numeric_cols)], remainder='passthrough')
scaling_transformation.fit(X_train)
X_train_scaled = scaling_transformation.transform(X_train)
print('Scaled shape:', X_train_scaled.shape)


Scaled shape: (504000, 19)


In [7]:
from sklearn.neighbors import KNeighborsClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

preprocessor_knn = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

knn_pipeline = ImbPipeline([
    ('preprocessing', preprocessor_knn),
    ('smote', SMOTE(random_state=50)),
    ('model', KNeighborsClassifier())
])

print(knn_pipeline)


Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['Soil_pH', 'Soil_Moisture',
                                                   'Organic_Carbon',
                                                   'Electrical_Conductivity',
                                                   'Temperature_C', 'Humidity',
                                                   'Rainfall_mm',
                                                   'Sunlight_Hours',
                                                   'Wind_Speed_kmh',
                                                   'Field_Area_hectare',
                                                   'Previous_Irrigation_mm']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Soil_Type', 'Crop_Type',
     

In [9]:

from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline as SkPipeline

preprocessor_xgb = ColumnTransformer(transformers=[
    ('num', 'passthrough', numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), cat_cols)
])

xgb_pipeline = SkPipeline([
    ('preprocessing', preprocessor_xgb),
    ('model', XGBClassifier(objective='multi:softmax', num_class=3, eval_metric='mlogloss', random_state=50, n_jobs=-1))
])

print(xgb_pipeline)


Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num', 'passthrough',
                                                  ['Soil_pH', 'Soil_Moisture',
                                                   'Organic_Carbon',
                                                   'Electrical_Conductivity',
                                                   'Temperature_C', 'Humidity',
                                                   'Rainfall_mm',
                                                   'Sunlight_Hours',
                                                   'Wind_Speed_kmh',
                                                   'Field_Area_hectare',
                                                   'Previous_Irrigation_mm']),
                                                 ('cat',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
       

In [10]:
X_train_lr = X_train.copy()
X_val_lr = X_val.copy()

X_train_lr['Water_Input'] = X_train_lr['Rainfall_mm'] + X_train_lr['Previous_Irrigation_mm']
X_train_lr['Evaporation'] = X_train_lr['Temperature_C'] * X_train_lr['Sunlight_Hours'] * X_train_lr['Wind_Speed_kmh']
X_train_lr['Moisture_Stress'] = X_train_lr['Temperature_C'] / (X_train_lr['Soil_Moisture'] + 1e-5)
X_train_lr['rain_temp'] = X_train_lr['Rainfall_mm'] * X_train_lr['Temperature_C']
X_train_lr['humidity_temp'] = X_train_lr['Humidity'] * X_train_lr['Temperature_C']

X_val_lr['Water_Input'] = X_val_lr['Rainfall_mm'] + X_val_lr['Previous_Irrigation_mm']
X_val_lr['Evaporation'] = X_val_lr['Temperature_C'] * X_val_lr['Sunlight_Hours'] * X_val_lr['Wind_Speed_kmh']
X_val_lr['Moisture_Stress'] = X_val_lr['Temperature_C'] / (X_val_lr['Soil_Moisture'] + 1e-5)
X_val_lr['rain_temp'] = X_val_lr['Rainfall_mm'] * X_val_lr['Temperature_C']
X_val_lr['humidity_temp'] = X_val_lr['Humidity'] * X_val_lr['Temperature_C']

new_cols = ['Water_Input', 'Evaporation', 'Moisture_Stress', 'rain_temp', 'humidity_temp']
scaler = MinMaxScaler().set_output(transform='pandas')
scaler.fit(X_train_lr[new_cols])
X_train_lr[new_cols] = scaler.transform(X_train_lr[new_cols])
X_val_lr[new_cols] = scaler.transform(X_val_lr[new_cols])
X_train_lr.head()


,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,...,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Water_Input,Evaporation,Moisture_Stress,rain_temp,humidity_temp
157685,Sandy,5.95,20.73,1.28,2.65,12.35,33.15,1764.14,4.53,7.40,...,Reservoir,6.79,No,62.53,South,0.696970,0.042750,0.081648,0.207919,0.029336
276639,Loamy,7.56,14.61,1.37,1.05,19.69,71.04,1746.91,5.13,1.18,...,Rainwater,14.80,No,47.89,East,0.684774,0.010028,0.231121,0.328336,0.297932
397859,Clay,6.22,48.94,0.76,1.84,41.09,85.02,522.42,5.92,2.95,...,Groundwater,8.25,Yes,113.06,Central,0.241146,0.076448,0.130119,0.204855,0.866600
243295,Silt,5.97,36.93,0.77,1.70,21.36,42.11,1551.64,4.02,15.21,...,River,13.93,No,7.42,South,0.594565,0.141763,0.078197,0.316364,0.162380
170481,Clay,6.06,8.52,1.07,1.17,28.68,53.93,849.38,9.94,1.15,...,Groundwater,2.88,No,36.25,West,0.336869,0.033187,0.632355,0.232491,0.338094


In [18]:
#transforming the categorical variable using one-hot encoding
from sklearn.preprocessing import OneHotEncoder

categorical_cols = [col for col in X_train.columns if X_train[col].dtype == 'object']
numeric_cols = X_train.columns.difference(categorical_cols)

# 2. Fit on TRAIN
encoder = OneHotEncoder(handle_unknown='ignore' ,drop='first', sparse_output=False)

train_X_cat = encoder.fit_transform(X_train[categorical_cols])

# 3. Transform TEST (no fit!)
test_X_cat = encoder.transform(X_test[categorical_cols])

# 4. Combine numerical and categorical columns
feature_names = encoder.get_feature_names_out(categorical_cols)

train_X_cat_df = pd.DataFrame(train_X_cat, columns=feature_names, index=X_train.index)
test_X_cat_df = pd.DataFrame(test_X_cat, columns=feature_names, index=X_test.index)

train_X_final = pd.concat([X_train[numeric_cols], train_X_cat_df], axis=1)
test_X_final = pd.concat([X_test[numeric_cols], test_X_cat_df], axis=1)

In [ ]:
#Exporting files
train_X_final.info()
train_X_final.to_csv("../data/train_test_data/train_X_final.csv", index=False)
test_X_final.to_csv("../data/train_test_data/test_X_final.csv", index=False)

In [19]:
mapping= {'Low': 0, 'Medium': 1, 'High': 2}

y_train_encoded = y_train.map(mapping)
y_test_encoded = y_test.map(mapping)
y_train_encoded.to_csv("../data/train_test_data/y_train_encoded.csv", index=False)
y_test_encoded.to_csv("../data/train_test_data/y_test_encoded.csv", index=False)